# 05 - Benchmarking and analysis of results

**Aim:** train the final von Heijne and SVM predictors, evaluate them on the independent benchmarking set and interpret their prediction errors.

We will reuse the prepared proteins and training folds from 02b, the von Heijne functions from 03 and the main 31-feature SVM procedure from 04. Parameter selection will use training data only. Once both predictors are fixed, we will compare their benchmarking predictions and analyse false positives and false negatives.

**Materials**
- `05 - Benchmarking and analysis of results.pdf` (slides 2–3: final models; 4–7: error analysis; 8: summary).
- [03 - Von Heijne method](03_von_heijne_method.ipynb) (sections 1–4: matrix, scoring and threshold; 5: metrics).
- [04 - SVM method](04_svm_method.ipynb) (sections 1–3: features and fitting; 4–5: fold-specific selection; 6: reporting).
- [02c - Data analysis](02c_data_analysis.ipynb) (composition, taxonomy and cleavage-site logos) and [04 feature exploration](04_feature_exploration.ipynb) (feature distributions).

**Status:** work guide only. Code cells are empty; final-model training, exports and benchmarking analysis have not been performed.

## Scope and notation

We will use the main SVM with 31 candidate features and retain the fixed 27-combination grid from 04. The experimental SVM variants remain separate. Existing CV results describe the prediction procedures; they do not supply a final model or a final benchmarking result (slides 2–3).

Our saved folds are numbered 1–5, corresponding to the five folds shown as 0–4 in slide 2. For von Heijne, folds 1–4 will train the threshold-selection matrix and fold 5 will validate its threshold. For SVM, all five folds will rotate through validation in a standard five-fold search.

We retain `n` for the composition-region length and `k` for the number of selected features. The regional “K” in slides 6–7 corresponds to our `n`. Benchmarking labels, metadata and error patterns must not guide feature design, parameter selection or threshold adjustment.

## 0. Imports and data loading

### 0.1 Imports and paths

Use the Conda `LB2` kernel and paths relative to `notebooks/`, following 04 section 0. Import the libraries needed for sequence handling, numerical tables, plotting, RF ranking, scaling, SVM fitting and metrics. Reuse the local WebLogo/Ghostscript setup from 02c when implementing the logo step.

**Inputs:** the four prepared FASTA/TSV files under `../data/prepared/`.

**To implement:** define `input_dir`, `table_dir = ../results/tables/05/` and `figure_dir = ../results/figures/05/` as relative paths, and create the output directories when this cell is implemented and run.

**Produce:** imports, path objects and plotting settings consistent with 03–04. New outputs belong to 05; earlier results remain the inputs and references.

### 0.2 Load sequences, labels and metadata

Reuse `read_fasta` and the accession-based loading in 04 section 0.2. Preserve `split`, `cv_fold`, `organism`, `kingdom`, `cleavage_pos` and `has_tm_in_90`; labels remain 1 for SP and 0 for NO_SP. Parse the TM flag as a boolean.

**Inputs:** `positive_dataset.tsv`, `negative_dataset.tsv` and their FASTA files.

**To implement:** build `prepared_data`, then separate `data` (training) and `benchmarking_data` using the saved split. Keep folds 1–5 unchanged (slide 2).

**Produce:** sequence/metadata tables for 8,146 training proteins (881 SP, 7,265 NO_SP) and 2,038 benchmarking proteins (221 SP, 1,817 NO_SP). These counts describe the existing inputs, not new model results.

## 1. Final von Heijne predictor

### 1.1 Estimate the threshold-selection matrix

Reuse `extract_cleavage_contexts`, `build_pswm`, `score_sequence` and `score_dataset` from 03 sections 1–3, defining them here without executing another notebook. Retain 15-position contexts (−13…−1,+1,+2), pseudocount 1, the fixed SwissProt background and scoring over complete windows in the first 90 residues.

**Inputs:** training folds 1–4 and validation fold 5.

**To implement:** estimate a new matrix from positive contexts in folds 1–4, then score both classes in fold 5. Do not use a three-fold matrix from 03.

**Produce:** a threshold-selection matrix, its training contexts and validation scores (slides 2–3).

### 1.2 Select the final threshold

Reuse `find_optimal_threshold` from 03 section 4 to maximize validation F1. Preserve the existing `score >= threshold` decision rule and the first maximum in a tie.

**Inputs:** fold-5 labels and scores from 1.1.

**To implement:** fix the threshold using only this validation fold and record its provenance. The professor mentioned selecting the threshold by F1 with this four-plus-one validation scheme.

**Produce:** the fixed threshold and selection record, planned as `1_2_von_heijne_threshold.tsv`. Validation F1 is a selection score, not final performance (slide 3).

### 1.3 Retrain the final matrix

**Inputs:** all positive proteins in `data` and the fixed threshold from 1.2.

**To implement:** rebuild the matrix using all five training folds, retaining the same context and scoring definitions. Keep the threshold unchanged after retraining (slide 3; also mentioned by the professor).

**Produce:** the final matrix and its context records, planned as `1_3_von_heijne_matrix.tsv` and `1_3_von_heijne_contexts.tsv`, together with the threshold record. No benchmarking prediction is needed for this step.

## 2. Final SVM predictor

### 2.1 Reuse feature extraction and define the search

Reuse the 31 feature definitions in 04 sections 1–2 and its `get_n`, `rank_features`, `fit_svm` and `predict_svm` helpers. Define the necessary functions and constants here so this notebook can run independently. Use 04's RF settings (500 trees, seed 0), StandardScaler and RBF SVC.

**Inputs:** training sequences, positive SP lengths and the feature definitions/catalog from 04.

**To implement:** retain `n = ceil(median positive cleavage_pos)` within each fitting subset. Define `k = 8, 16, 31`, `C = 1, 4, 16` and `gamma = factor/k` for factors 0.5, 1, 2. Existing sequence-only training caches for n=21 and n=22 may be read from `../results/tables/04/feature_cache/` when compatible; derive each required n before using a cache. Adapt 04's training-only cache interface for new subsets rather than using it unchanged for benchmarking.

**Produce:** independent feature/fitting helpers and the ordered 27-candidate grid. These are the agreed project settings, not a new grid specified by slide 3.

### 2.2 Select settings by five-fold cross-validation

**Inputs:** all five saved training folds and the grid from 2.1.

**To implement:** rotate each fold through validation, fitting on the other four. For each fitting subset, derive n and fit its RF ranking; for each candidate, select its top-k features and fit the scaler/SVM on that subset only. Select `(k, C, gamma)` by mean validation MCC across all five folds; exact ties retain the first grid candidate.

**Produce:** fold/candidate selection records and the selected configuration, planned as `2_2_svm_search.tsv` and `2_2_svm_settings.tsv` (slides 2–3; the professor mentioned selecting by mean MCC).

This search replaces the outer-test loop for final selection. Do not reuse 04's inner scores as this five-fold search or choose an outer-fold winner. Keep search scores as selection records, separate from reported performance.

### 2.3 Refit and export the final predictor

**Inputs:** all training proteins and the selected configuration from 2.2.

**To implement:** derive the final n from all training positives, extract the features, fit a fresh RF ranking, retain the selected k names, and refit the scaler and SVM on all training proteins. Preserve the selected feature order when predicting.

**Produce:** the final predictor, final n, selected feature names/ranks/importances and settings. Plan exports as `2_3_svm_selected_features.tsv` and `2_3_svm_final.pkl` (scaler, fitted SVM, ordered names, n and settings). The feature functions remain defined in this notebook. No final fitted predictor is currently saved by 04 (slide 3).

## 3. Benchmarking evaluation

### 3.1 Generate final predictions

**Inputs:** the fixed matrix/threshold from 1.3, the fixed SVM predictor from 2.3 and `benchmarking_data`.

**To implement:** score/classify the same benchmarking proteins once with each final predictor. For SVM, compute benchmark features using the final training-derived n and apply the saved feature order and scaler. Labels and annotations are used for evaluation only.

**Produce:** `3_1_benchmark_predictions.tsv` with accession, model, actual label, prediction and von Heijne score where applicable. Retain metadata through accession joins for subsequent analysis (slides 2–4).

### 3.2 Compare metrics and confusion matrices

Reuse the metric calculations and confusion-plot conventions in 03 section 5 and 04 section 6.

**Inputs:** the two sets of benchmarking predictions.

**To implement:** calculate TP, FN, FP, TN and report MCC, F1, precision, recall and accuracy side by side. Plot confusion counts and percentages within each actual class.

**Produce:** `3_2_benchmark_metrics.tsv` and a numbered confusion-matrix PDF under `../results/figures/05/`. Each metric is a single benchmarking estimate; do not attach the CV standard errors or average this evaluation over the training folds (slide 4; the professor mentioned reporting a single estimate for each model).

## 4. False-positive analysis

### 4.1 Compare global and TM false-positive rates

**Inputs:** benchmarking predictions for both methods and negative metadata `has_tm_in_90`. This UniProt-derived flag already records whether a TM region starts within residues 1–90; the numerical TM-tendency feature is not a substitute for this annotation.

**To implement:** isolate each model's FP predictions and calculate:
- TM fraction among FP: `FP_TM / FP` (slide 5).
- Overall false-positive rate: `FP / (FP + TN)`.
- TM-restricted false-positive rate: `FP_TM / (FP_TM + TN_TM)`.

**Produce:** `4_1_fp_tm_analysis.tsv` with counts, denominators and fractions/rates for both models, plus a comparative rate plot. If a denominator is zero, report the rate as undefined.

The professor mentioned comparing the overall and TM-restricted false-positive rates because the fraction among FP alone does not account for TM abundance. Interpret any enrichment as an association; annotations alone do not prove the cause of an individual error.

## 5. False-negative analysis

### 5.1 Compare von Heijne TP and FN cleavage-site logos

**Inputs:** von Heijne benchmark TP/FN accessions, sequences and annotated cleavage positions. The existing training logo from 02c provides the expected motif reference.

**To implement:** extract the annotated 15-residue contexts for TP and FN and reuse 02c section 4's local WebLogo/Ghostscript procedure. Keep the same alphabet, colors, position labels and shared axis for the new comparison; report the number of contexts in each group.

**Produce:** group-specific context FASTA files under `../results/tables/05/` and TP/FN logo PDFs under `../results/figures/05/`.

Slide 6 requests an FN logo compared with the expected logo; the professor mentioned comparing TP and FN logos. Discuss motif differences descriptively and consider small FN sample sizes. Lower FN conservation is a hypothesis, not an observed result.

### 5.2 Compare SVM N-terminal composition

**Inputs:** positive training proteins, SVM benchmark TP/FN groups and the final n from 2.3.

**To implement:** calculate per-protein amino-acid composition within the first n residues and compare group summaries using the composition conventions in 02c and 04. Use the same n for all three groups. These are sequence compositions, not standardized SVM inputs.

**Produce:** `5_2_svm_composition.tsv` and a three-group composition PDF (slides 6–7). Explain any differences without changing n, features or the fitted model.

### 5.3 Compare SP lengths and selected features

**Inputs:** the same three positive groups, `cleavage_pos`, final n and the selected feature names from 2.3.

**To implement:** compare annotated SP-length distributions and the original-unit distributions of the selected SVM features. Reuse the feature extraction and plotting conventions from 04 and its supplementary exploration. Training positives are the reference group, not held-out predictions. Composition descriptors already shown in 5.2 need not be plotted again.

**Produce:** group counts and descriptive summaries in `5_3_svm_fn_features.tsv`, with numbered distribution PDFs (slides 6–7). The professor mentioned focusing on features selected in the final refit and gave a hydrophobicity boxplot as an example.

Describe overlap and differences without adding significance tests or redesigning the predictor after inspecting benchmarking errors.

## 6. Taxonomic errors and interpretation

### 6.1 Compare error profiles across taxa and species

**Inputs:** both methods' benchmark predictions and the existing `kingdom` and `organism` metadata.

**To implement:** count FP and FN by taxonomic group and species. Include the corresponding negative and positive totals, reporting `FP / N_negative` and `FN / N_positive` where defined. Use the taxonomy presentation conventions in 02c and keep small-group counts visible.

**Produce:** `6_1_taxonomic_errors.tsv` and concise comparative figures/tables (slide 4). Describe whether observed errors concentrate in particular groups while accounting for their representation; raw counts alone do not establish a higher error rate.

### 6.2 Interpretation and limitations

**To complete after execution:** compare the two models using the benchmarking metrics from 3.2, then interpret their TM-associated false positives, FN patterns and taxonomic error profiles (slides 4–8). Refer to the exported tables and figures and distinguish observations from possible explanations.

Keep 03–04 CV estimates as evaluations of the earlier procedures, final-search scores as parameter-selection records, and benchmarking metrics as the independent final comparison. Do not present search scores as an improved performance estimate or use benchmark errors to select a different model. The professor mentioned discussing these limitations in the report; later model improvements are outside this section.

**Expected output:** a short evidence-based account of the models' strengths and limitations. No results or conclusions are available yet.